In [1]:
import torch
import torch.nn as nn

class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size=1000, embed_dim=64, hidden_dim=128, num_classes=2):
        super(SentimentLSTM, self).__init__()
        
        # 1. Embedding Layer: Har Token ID ko 64-dim continuous vector me badlega
        # Table shape: [1000, 64]
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)
        
        # 2. LSTM Engine: 64 numbers lega, 128 numbers ka hidden memory banayega
        self.lstm = nn.LSTM(
            input_size=embed_dim,     # 64
            hidden_size=hidden_dim,   # 128
            batch_first=True          # batch dimension pehle aayegi
        )
        
        # 3. Final Linear Layer: 128 hidden numbers ko 2 classes (Positive/Negative) me map karega
        self.fc = nn.Linear(hidden_dim, num_classes)
        
    def forward(self, x):
        # Input 'x' shape: [batch_size=2, seq_len=5] -> (2 sentences, 5 words each)
        
        # Step A: Lookup Embeddings
        # out shape: [2, 5, 64]
        embedded = self.embedding(x)
        
        # Step B: Pass through LSTM loop
        # out: [2, 5, 128] -> Har word ka hidden state
        # h_n: [1, 2, 128] -> Bilkul aakhri (5th) word ka final hidden state (context)
        # c_n: [1, 2, 128] -> Aakhri word ka cell state
        lstm_out, (h_n, c_n) = self.lstm(embedded)
        
        # Step C: Humein pure sentence ka ek final context chahiye
        # h_n shape [1, 2, 128] -> squeeze karke [2, 128] banaya
        final_context = h_n.squeeze(0)
        
        # Step D: Classification layer
        # logits shape: [2, 2] (Positive vs Negative score)
        logits = self.fc(final_context)
        
        return logits

# --- Run Karke Dekhte Hain ---
model = SentimentLSTM()

# Dummy input: 2 sentences, 5 word IDs each
sample_data = torch.tensor([
    [10, 45, 230, 89, 5],   # Sentence 1: "Movie was really good actually"
    [7, 18, 901, 12, 66]    # Sentence 2: "Worst acting and bad plot"
])

output = model(sample_data)
print("Output Predictions (Logits):\n", output)
print("Shape:", output.shape) # [2, 2]

Output Predictions (Logits):
 tensor([[ 0.1766,  0.0137],
        [ 0.1448, -0.1829]], grad_fn=<AddmmBackward0>)
Shape: torch.Size([2, 2])
